# Phase 6.5 — volatility feature comparison

This notebook tests whether the two approved volatility inputs help. It compares all 14 approved market inputs with exactly the same set **without volatility**. Return and risk are checked separately across the same four historical time folds.

It fits models. Locally, select the project `.venv-phases` kernel. In Colab, the setup cell clones the repository, restores the verified Phase 5 input, and installs pinned dependencies. It writes only to the separate Phase 5-continuation evidence folder and refuses to overwrite an existing result.

In [ ]:
from pathlib import Path
import subprocess
import sys

BRANCH = 'feature/model-evaluation'
COLAB_REPOSITORY = Path('/content/HalalifyAI')
if 'google.colab' in sys.modules:
    if not (COLAB_REPOSITORY / '.git').is_dir():
        subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', 'https://github.com/frager098/HalalifyAI.git', str(COLAB_REPOSITORY)], check=True)
    root = COLAB_REPOSITORY.resolve()
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(root / 'requirements.txt'), '-r', str(root / 'requirements-cleaning.txt'), 'gdown>=5.2,<6'], check=True)
else:
    root = Path.cwd().resolve()
    while root != root.parent and not (root / 'src' / 'model_training').is_dir():
        root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from src.notebook_environment import ensure_phase5_dataset, prepare_repository
root = prepare_repository(branch=BRANCH, colab_repository=COLAB_REPOSITORY)
ensure_phase5_dataset(root)
%cd {root}
print(f'Project folder: {root}')

## Run the controlled comparison

The next cell uses only the exact dataset and chronological protocol continued from Phase 5. It tunes only on earlier inner-validation data, then scores each later outer fold once. Expect a few minutes of work.

In [ ]:
from src.model_training.feature_ablation.phase5_continuation import (
    VOLATILITY_OUTPUT,
    prepare_inputs,
    write_volatility_report,
)

prepare_inputs()
report = write_volatility_report()
print('Status:', report['status'])
print('Evidence folder:', VOLATILITY_OUTPUT)
for task, summary in report['task_summaries'].items():
    print(f"{task}: {summary['decision']} | {summary['primary_metric']} difference = {summary['mean_difference']:.4f}")

## Read the result

`retain` means volatility consistently helped. `reject` means it did not help in most time periods. `inconclusive` means the evidence was mixed. For risk, a result cannot be retained if High-risk detection becomes worse or if more High-risk cases are wrongly predicted as Low risk.

In [ ]:
for task, summary in report['task_summaries'].items():
    print('\n', task.upper())
    display(summary['fold_metrics'])
    if task == 'risk':
        print('Risk safety checks:', summary['risk_guardrails'])